# Solutions · Chapter 00-02 · What ML is, what it is not, and when a rule wins

Worked answers with the reasoning, the mistake each exercise was built to catch, and code
where it applies. Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.tree import DecisionTreeClassifier

# Task A, exactly as in the chapter.
rng = np.random.default_rng(4)
n_orders = 300
order_total = np.round(rng.uniform(5, 120, n_orders), 2)
is_domestic = rng.random(n_orders) < 0.70
orders = pd.DataFrame({"order_total": order_total,
                       "is_domestic": is_domestic.astype(int),
                       "free_delivery": (order_total >= 50) & is_domestic})
features = ["order_total", "is_domestic"]
train, test = orders.iloc[:200], orders.iloc[200:]
print("task A ready:", len(train), "train /", len(test), "held out")

In [ ]:
# Task B, exactly as in the chapter.
rng_b = np.random.default_rng(7)
n_parcels = 800
distance_km = rng_b.uniform(1, 900, n_parcels)
weight_kg = rng_b.uniform(0.1, 20, n_parcels)
holiday_week = rng_b.random(n_parcels) < 0.20
bad_weather = rng_b.random(n_parcels) < 0.30
pressure = (-3.9 + 0.004 * distance_km + 0.10 * weight_kg
            + 1.3 * holiday_week + 1.0 * bad_weather + rng_b.normal(0, 0.8, n_parcels))
parcels = pd.DataFrame({"distance_km": distance_km.round(1), "weight_kg": weight_kg.round(2),
                        "holiday_week": holiday_week.astype(int),
                        "bad_weather": bad_weather.astype(int), "late": pressure > 0})
parcel_features = ["distance_km", "weight_kg", "holiday_week", "bad_weather"]
p_train, p_test = parcels.iloc[:600], parcels.iloc[600:]
print("task B ready:", len(p_train), "train /", len(p_test), "held out")

## E1 · What ML inverts, and what it needs

> Ordinary programming takes **rules plus data** and produces **answers**; machine learning
> takes **data plus the answers** and searches for a rule that would have produced them. The
> extra ingredient it requires is recorded outcomes - examples labelled with what actually
> happened.

**The trap:** answering "it needs lots of data". Volume matters, but the thing that is
genuinely extra compared with writing a rule is the *labels*. Plenty of organisations have
enormous datasets and no column saying what the right answer was.

## E2 · A perfect score is not a reason to deploy

> The tree scored 1.000 because it recovered a rule we already possessed exactly, so the best
> possible outcome of the whole exercise was to draw level with three lines of code. To get
> there it consumed 200 labelled orders, a training step and a fitted object that now has to be
> stored, versioned and monitored. And the moment the policy changes it becomes silently wrong,
> while the three lines would have been a one-number edit.

**The trap:** treating the score as the decision. A score is one input to a decision that also
includes maintenance, explainability, latency, legal exposure and the cost of being wrong in a
way nobody notices.

## E3 · Accuracy, and where it breaks

**Accuracy** is the share of predictions that are correct: correct predictions divided by total
predictions. Dimensionless, usually quoted as a proportion or a percentage.

For a rare disease present in, say, 1 patient in 1000, the rule "nobody has it" scores
**0.999**. It is nearly perfectly accurate and completely useless - it never finds a single
patient, which is the entire purpose. Accuracy hides this because it counts the 999 easy
correct answers and the 1 catastrophic miss as though they were the same kind of event.

The habit to build now: **always ask what the majority-class baseline scores.** If your model's
accuracy is close to it, the model has found almost nothing, no matter how impressive the
number looks. Chapter 06-04 and 06-05 replace accuracy with measures that survive imbalance.

## E4 · Applying the new policy by hand

New policy: at least 40 EUR **and** domestic.

| Order | New policy says | The old model (boundary at 50) says | Agree? |
|---|---|---|---|
| (a) 39.99 domestic | no - one cent short | no | yes |
| (b) 40.00 domestic | **yes** - "at least" includes 40.00 | no | **wrong** |
| (c) 40.00 international | no - wrong country | no | yes |
| (d) 250.00 international | no - size never overrides country | no | yes |
| (e) 45.00 domestic | **yes** | no | **wrong** |

Two of the five wrong, and both wrong in the *same* place: domestic orders between 40 and 50
EUR. That is the signature of this kind of failure - errors are not scattered, they are
concentrated in a band, and everyone in that band is a customer who was promised something and
did not get it.

**The point of (c) and (d):** a large order does not compensate for the wrong country. The
policy is an AND, not a score. Models that learn a smooth trade-off between features get this
kind of hard constraint wrong at the edges, which is one reason hard constraints are usually
better encoded than learned.

## E5 · Accuracy from counts

Sort the 200 parcels into the four boxes first. This is a confusion matrix, and chapter 06-04
builds it properly - you can already fill it in.

- Predicted late and truly late: **45**
- Predicted late but actually on time: `60 - 45 =` **15**
- Predicted on time but truly late: **30** (given)
- Predicted on time and truly on time: `200 - 45 - 15 - 30 =` **110**

`accuracy = (45 + 110) / 200 = 155 / 200 = 0.775`

Actually late = `45 + 30 = 75`, so on time = 125.
`always-on-time accuracy = 125 / 200 = 0.625`

**Conclusion:** the model is 15 percentage points better than refusing to think - a genuine
improvement, not a rounding error, and worth taking seriously.

**What you would still need before recommending it:** the relative cost of the two mistakes.
The model raises 15 false alarms and misses 30 genuinely late parcels. If a missed late parcel
means an angry customer and a refund, while a false alarm means someone glances at a dashboard,
then those 30 misses dominate the decision and you would want to move the threshold to catch
more of them, accepting more false alarms. Accuracy is blind to that trade entirely - it
weights the 15 and the 30 the same. Chapter 06-07 does this properly.

In [ ]:
tp, fp, fn = 45, 15, 30
tn = 200 - tp - fp - fn
print("confusion counts  TP", tp, " FP", fp, " FN", fn, " TN", tn)
print("model accuracy         ", (tp + tn) / 200)
print("always-on-time accuracy", (tn + fp) / 200)

## E6 · How much data does it take to rediscover a rule you already had?

In [ ]:
sizes = [10, 20, 50, 100, 200]
accs = []
for k in sizes:
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(
        train[features].iloc[:k], train["free_delivery"].iloc[:k])
    accs.append(accuracy_score(test["free_delivery"], t.predict(test[features])))
    print(f"trained on {k:3d} orders -> accuracy {accs[-1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(sizes, accs, marker="o", color="#0072B2", label="decision tree")
ax.axhline(1.0, color="#D55E00", linestyle="--", label="the written policy (no data)")
ax.set_xlabel("Number of labelled training orders")
ax.set_ylabel("Accuracy on 100 held-out orders")
ax.set_ylim(0.65, 1.03)
ax.set_title("Data buys you a rule you already had")
ax.legend()
plt.show()

**The shape:** 0.710 with 10 orders, 0.860 with 20, 0.960 with 50, and 1.000 from 100 onwards.
Steep, then flat.

Three things the curve tells you, and all three generalise far beyond this toy:

1. **Early data is worth much more than late data.** The first forty examples bought 25
   percentage points; the second hundred bought nothing at all. "Get more data" is excellent
   advice on the steep part and a waste of money on the flat part - and you cannot tell which
   part you are on without drawing this curve. It is called a **learning curve**, and 05-08 and
   07-02 make it a diagnostic tool.
2. **There is a ceiling, and here the ceiling is the policy.** No amount of data takes the model
   above the thing it is imitating. Whenever a model is learning something that already exists
   in exact form, its best case is a tie.
3. **The dashed line needed no data at all.** On the left of this plot, the written rule is
   beating the model by thirty points, for free. That gap is what you give away when you skip
   question 2 of the seven.

## E7 · A better hand rule for task B

In [ ]:
grid_a = np.arange(100, 900, 20)
grid_b = np.arange(50, 900, 20)

best = (0.0, None, None)
for a in grid_a:                      # chosen on TRAINING data only
    for b in grid_b:
        pred = (p_train["distance_km"] > a) | ((p_train["bad_weather"] == 1) & (p_train["distance_km"] > b))
        s = accuracy_score(p_train["late"], pred)
        if s > best[0]:
            best = (s, a, b)

_, a, b = best
two_rule = (p_test["distance_km"] > a) | ((p_test["bad_weather"] == 1) & (p_test["distance_km"] > b))
model = LogisticRegression(max_iter=1000).fit(p_train[parcel_features], p_train["late"])

print(f"best two-condition rule: distance > {a} OR (bad weather AND distance > {b})")
print(f"  baseline 'always on time'   {accuracy_score(p_test['late'], np.zeros(len(p_test), bool)):.3f}")
print(f"  one-condition rule          0.730")
print(f"  this two-condition rule     {accuracy_score(p_test['late'], two_rule):.3f}")
print(f"  logistic regression         {accuracy_score(p_test['late'], model.predict(p_test[parcel_features])):.3f}")

**The result:** `distance > 640 OR (bad weather AND distance > 370)` scores **0.770** on the
held-out parcels, against 0.730 for the single condition and 0.855 for the model.

So the second condition was worth four points. Progress - and also the answer to the exercise.

**When hand rules stop being worth writing.** Notice what that four points cost: a search over
roughly 1,600 combinations of two numbers, which is not something a person does in their head.
And there are still two features - weight and holiday week - completely unused, each of which
would multiply the search again. What we were doing by hand *is* what fitting a model does; we
were simply doing it slowly, for a smaller space of rules.

That is the honest boundary between the two approaches. Rules win while a person can hold the
whole rule in their head and defend each number in it. Once you find yourself grid-searching
thresholds, you have already started doing machine learning - you may as well use a method
designed for it, which will search a richer space and give you the validation machinery for
free.

**Careful, though:** we chose `a` and `b` using the training set and reported on the held-out
set. Choosing them on the test set instead would have produced a higher number and a worse
rule - the same mistake as scoring on data you fitted to, one level up. Chapter 07-04 is
entirely about that trap.

## E8 · "94% model versus 91% rules"

Three questions, in order of how likely they are to change the decision:

1. **What does the majority-class baseline score?** If 92% of customers do not churn, then both
   systems are *worse than refusing to think*, and the comparison is meaningless. This question
   costs ten seconds and has ended many such discussions.
2. **Were both measured the same way, on the same held-out customers, over the same period?**
   A model evaluated on a random split and a rule evaluated on last quarter's live traffic are
   not comparable numbers. Also: was the model tuned on the data it was scored on?
3. **What is the cost of each kind of error, and are the two systems making the same kind?**
   The model may be three points better overall while missing exactly the high-value customers
   the rules were written to catch. Aggregate accuracy would hide that completely.

A good fourth: **what does it cost to run and change each?** If the rules can be updated by the
retention team in an afternoon and the model needs a data scientist and a retraining pipeline,
three points may not be worth it.

## E9 · Accuracy slipped from 0.88 to 0.79

Ordered from cheapest to check to most expensive - and the ordering is the answer, because the
instinct being trained is *"suspect the measurement before you suspect the world"*.

1. **The measurement changed, not the model.** A reporting bug, a changed join, a different
   population in the monthly report (new customer segment onboarded), or labels arriving later
   than they used to so recent months look artificially bad. **Check:** recompute last month's
   number from raw data yourself, and recompute an *old* month with today's pipeline - if the
   old month no longer reproduces 0.88, the pipeline moved, not the world.
2. **The input data changed.** An upstream field is now null, a unit changed, a category was
   renamed, a default value replaced a real one. **Check:** compare the distribution of every
   input feature this month against the training period - counts of nulls, means, category
   frequencies. This is the cheapest genuine drift check there is, and 13-08 formalises it.
3. **The world changed (concept drift).** Payment terms, customer mix, an economic shift - the
   relationship between features and lateness genuinely moved. The model is faithfully
   describing a world that no longer exists.
4. **The model was never as good as 0.88.** The original figure may have been optimistic:
   tuned on the test set, evaluated on a random split when it should have been chronological, or
   measured on an unusually easy period. Eight months of honest measurement is often the first
   *real* estimate a model ever gets.

**The instinct:** a metric moving is a statement about a measurement pipeline until proven
otherwise.

## E10 · "When would you not use machine learning?" - a spoken answer

> I would not use it when I can write the rule down. If a policy exists - a fee threshold, a
> validation format, a regulation - encoding it gives an exact answer, an explanation and a
> one-line change when the policy moves, where a model can at best imitate it and will smooth
> its edges. I would not use it when the question is about the past: "how much did we sell in
> July" is a query, and an estimate of a recorded fact is a downgrade. I would not use it when
> the question is causal - "does moving the display increase sales" needs an experiment,
> because a predictive model will happily learn that umbrella sales predict rain. And I would
> not use it when we do not have recorded outcomes, when the decision happens four times a year,
> or when a single wrong answer is unacceptable and unexplainable. What is left after all that
> is where ML genuinely earns its keep: unknown rule, plenty of labelled examples, high volume,
> survivable mistakes.

(About 150 words. It uses questions 2, 3, 5, 6 and the "past versus future" and "predict versus
cause" distinctions.)

**What is actually being assessed:** whether you have a *framework* or a *list*. Candidates who
say "when you have little data" have one memorised fact. Candidates who ask what the decision
is, how often it happens and what a mistake costs are demonstrating the thing the job needs.

## E11 · Loan approval: which parts should be rules?

1. **Hard regulatory and policy constraints must be rules.** Minimum age, sanctions screening,
   maximum loan-to-income ratios set by a regulator. These are known exactly and must hold in
   *every* case, not in 99.4% of cases. **Question 2** (the rule is writable) and **question 5**
   (a single wrong answer is not acceptable).
2. **Decisions must be explainable to the applicant, and often legally justifiable.** In many
   jurisdictions a declined applicant is entitled to a reason. A rule states its reason; a model
   needs extra machinery to approximate one, and the approximation is not the decision.
   **Question 5.**
3. **The training data records who was approved and how they behaved - not how rejected
   applicants would have behaved.** You never observe the outcome for people the old system
   declined, so the data is a record of past policy as much as of repayment. **Question 3**
   (are the outcomes really recorded?) and **question 4** (does the future resemble the past -
   here, the *past includes the system you are replacing*).

A strong answer adds: the sensible architecture is usually rules for eligibility and a model
for risk ranking *within* the eligible set, so the constraints stay exact and the learning
happens where the rule is genuinely unknown. That is the hybrid from the remedies table, and
E14 builds a small version of it.

## E12 · Rule, query, experiment, optimisation, or ML?

**(a) Which warehouse ships this order to minimise total distance, given stock levels?**
**Optimisation.** Nothing is unknown - you have the stock levels, the addresses and the
distances, and you want the best assignment under constraints. Predicting the answer would be
strictly worse than computing it.

**(b) Will this customer open the email we are about to send?**
**ML.** The rule is unknown, outcomes are recorded automatically every time you send an email,
the volume is enormous and a wrong guess costs almost nothing. This is the textbook case.

**(c) Did last quarter's price increase reduce order volume?**
**Experiment - and if you cannot run one, a causal method, carefully.** It is a question about
an intervention that already happened, and everything else moved at the same time (season,
competitors, marketing). A predictive model fitted to this data will confidently attribute the
change to whatever correlates with it. 12-07.

**(d) Is this passport number in a valid format?**
**A rule.** There is a published specification with a checksum. Exact, free, and a model would
be wrong on the rare formats that matter most.

**(e) How many support agents on shift next Tuesday at 14:00?**
**Both, in order.** The prediction of contact volume is ML (or a good seasonal baseline - see
09-04, and start there). Turning that prediction into a staffing decision is optimisation, under
constraints like shift lengths and contracts. Notice how often the real task decomposes like
this: predict the uncertain quantity, then optimise the decision that uses it.

## E13 · Explaining it to Tomás

> Your till knows what people spent. It does not know why you gave anyone a discount - that
> came from the card behind your counter, and the card is not in the till. So a computer looking
> at your sales can only guess at the policy by watching who happened to get it, and it will get
> the edges wrong: someone who spent 99.50 and 100.20 look almost identical to it. You already
> have the exact answer. Use it.

(66 words, no "model", "algorithm" or "data".)

**The instinct:** the person asking for ML usually has a clearer statement of the rule than any
model could recover - it is just written somewhere that is not the database. Ask them to read it
to you before you write any code.

## E14 · The hybrid: exact where you know, learned where you do not

The fee is decided by the policy. The delivery **duration** genuinely is unknown - it depends on
distance, weather and things nobody records - so that part is learned.

In [ ]:
# Duration in days, SYNTHETIC: driven by distance and weather, plus noise.
duration_days = np.clip(0.8 + 0.006 * distance_km + 0.9 * bad_weather
                        + rng_b.normal(0, 0.5, n_parcels), 0.5, None).round(2)
deliveries = parcels.assign(days=duration_days)
d_train, d_test = deliveries.iloc[:600], deliveries.iloc[600:]

duration_model = LinearRegression().fit(d_train[["distance_km", "bad_weather"]], d_train["days"])
pred_days = duration_model.predict(d_test[["distance_km", "bad_weather"]])

print(f"learned part - MAE {mean_absolute_error(d_test['days'], pred_days):.2f} days")
print(f"  vs baseline 'always the average' {mean_absolute_error(d_test['days'], [d_train['days'].mean()] * len(d_test)):.2f} days")

In [ ]:
def quote(order_total, is_domestic, distance_km, bad_weather, free_from=50):
    """Fee comes from the policy (exact). Duration comes from the model (estimated)."""
    fee = 0.0 if (order_total >= free_from and is_domestic) else 4.95
    row = pd.DataFrame([{"distance_km": distance_km, "bad_weather": bad_weather}])
    days = float(duration_model.predict(row)[0])
    return fee, round(days, 1)

print("45 EUR domestic, 300 km, clear   ->", quote(45, True, 300, 0))
print("  after the policy moves to 40   ->", quote(45, True, 300, 0, free_from=40))
print("45 EUR international, 300 km     ->", quote(45, False, 300, 0, free_from=40))

**What the hybrid bought you.** The fee is exact and stays exact: changing the policy is one
keyword argument, takes effect instantly, needs no data and no retraining, and is explainable to
a customer in one sentence. The duration estimate - MAE 0.43 days against a baseline of 1.44 -
is genuinely learned, because nobody could have written that rule down.

**What it cost.** Two components instead of one, so two things to test, two things to version,
and a seam between them where mistakes hide: if the policy changes and the duration model was
trained on data where the old fee affected customer behaviour, the learned half can still go
stale even though the rule half is fine. Hybrids are the right answer far more often than they
are the easy one.

**The pattern to carry forward:** *encode what you know, learn what you do not, and keep the
seam between them visible.* You will meet it again in 13-04, where packaging a pipeline is
mostly about making sure that seam behaves the same in training and in production.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **00-03 · The kinds of learning,
and the map of the work**.